# Day 1: Trade data and Morocco's complexity profile

ECU Summer School of Economic Complexity. UM6P Rabat. Afternoon Session 1.

Today we take a country from raw export values to its economic-complexity profile by hand. We do not call a complexity package yet. That comes later in the week, once you have seen the inner workings and the index is no longer a black box.

The whole afternoon follows one five-step skeleton, reused on Days 2 and 3.

> Normalize → Benchmark → Binarize → Count → Iterate

At the end we compare our hand-rolled numbers to WIPO's published figures and work out why they differ. To make the notebook about your own country, edit `COUNTRY` and `COMPARATORS` at the top. They drive every figure below.

If you are reading this on Google Colab, save your own copy before you start: open the File menu and choose "Save a copy in Drive". Colab discards the session when it closes, and without your own copy your work goes with it.

## 0. Setup: load the data packet

This section gets everyone past the file-loading friction, with nothing conceptual yet. The notebook finds the data packet on its own. It uses the course repository's copy when you work from a local checkout, and otherwise downloads the packet (about 45 MB) from the course GitHub release. On Colab the download runs automatically the first time you execute the cell, so there is nothing to install or upload.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import spearmanr

# --- The ONE place you edit to make this notebook about your own country. ------
# Codes are ISO 3166-1 alpha-2 (two letters). MA = Morocco, EG = Egypt,
# TN = Tunisia, KR = Republic of Korea, DE = Germany. See reference/units.csv.
COUNTRY = "MA"  # the focus country (Morocco)
COMPARATORS = ["EG", "TN", "KR"]  # Egypt, Tunisia (MENA) + Korea (frontier)
YEAR = 2023  # latest year in the packet
# -------------------------------------------------------------------------------

# Resolve the data packet. Two cases, checked in order:
#   1. You cloned the course repository: use its local packet copy directly.
#   2. Anywhere else (Google Colab, or a fresh machine): download the packet
#      folders this notebook needs from the course GitHub release, once, into
#      a morocco_data_packet folder next to the notebook.
LOCAL_PACKET = Path("../../data/processed/morocco_data_packet")
DATA_RELEASE = "https://github.com/shreyasgm/ecu-complexity-labs/releases/download/data-v1"

if LOCAL_PACKET.exists():
    PACKET = LOCAL_PACKET.resolve()
else:
    PACKET = Path("morocco_data_packet").resolve()
    for folder in ("exports", "reference"):
        if not (PACKET / folder).exists():
            # About 45 MB in total; well under a minute on most connections.
            print(f"Downloading {folder}.zip from the course data release ...")
            zip_file, _ = urllib.request.urlretrieve(f"{DATA_RELEASE}/{folder}.zip")
            with zipfile.ZipFile(zip_file) as zf:
                zf.extractall(PACKET)

# Fail LOUD if the data still is not here (for example, the download was
# interrupted). A half-present packet is worse than a missing one, so name
# the exact folder that must exist before going on.
for folder in ("exports", "reference"):
    if not (PACKET / folder).exists():
        raise FileNotFoundError(
            f"Data packet folder not found at: {PACKET / folder}\n"
            "Re-run this cell to retry the download, or fetch the packet "
            f"zips yourself from {DATA_RELEASE} and unzip them into "
            f"{PACKET}. The notebook cannot run without them."
        )

print(f"Packet found: {PACKET}")
print(f"Focus country: {COUNTRY} | comparators: {COMPARATORS} | year: {YEAR}")

We use the Production and exports dimension today, which lives in the packet's `exports/` folder. That keeps a clean trade through-line into the Day 2 product space and the Day 3 density regressions. The packet also carries patents, publications, and trademarks, which we hold for Day 5.

In [ ]:
# Load the two tables we build everything from today.
#   outputs.parquet : raw export VALUE (current USD) per country x product x year
#   fields.csv      : product names + domains, for labels and the treemap
exports = pd.read_parquet(PACKET / "exports" / "outputs.parquet")
fields = pd.read_csv(PACKET / "reference" / "fields.csv")
units = pd.read_csv(PACKET / "reference" / "units.csv")

# The packet stores its measure columns as float32 to keep the download small.
# Upcast them to float64 before computing: summing hundreds of float32 values
# loses the last digits, and the self-checks below assert identities to 1e-9.
float32_cols = exports.select_dtypes("float32").columns
exports[float32_cols] = exports[float32_cols].astype("float64")

print("exports columns:", list(exports.columns))
print("exports rows:", f"{len(exports):,}")
exports.head(3)

## Warm-up: the whole pipeline on a toy table

We run the five steps on a table small enough to check by eye before scaling to the real matrix. These are the three countries and four products from the morning lecture (wheat, textiles, motorcycles, cell phones), with the same raw export values. Watch them become shares, then RCA, then a binary Mcp, then diversity and ubiquity. The real matrix later does the same thing at 191 countries by 862 products.

In [ ]:
# The morning lecture's toy example (from the Linear Algebra slides): the same
# 3 countries x 4 products of raw export values, small enough to verify by hand.
# One row per country, so this literal reads like the printed table below.
# fmt: off
toy = pd.DataFrame(
    [[ 10,  40, 100, 300],   # Country A
     [150, 100,  10,   0],   # Country B
     [ 20, 200,   1,   0]],  # Country C
    index=["Country A", "Country B", "Country C"],
    columns=["Wheat", "Textiles", "Motorcycles", "Cell phones"],
).astype(float)
# fmt: on

# Normalize then Benchmark: each country's share of its own basket, divided by the
# world's share of that product. The same two divisions the real matrix uses.
toy_shares = toy.div(toy.sum(axis=1), axis=0)
toy_world = toy.sum(axis=0) / toy.sum().sum()
toy_rca = toy_shares.div(toy_world, axis=1)

# Binarize: a product is present (1) where RCA > 1.
toy_mcp = (toy_rca > 1).astype(int)

# Count: diversity across each row, ubiquity down each column.
toy_diversity = toy_mcp.sum(axis=1)
toy_ubiquity = toy_mcp.sum(axis=0)

print("RCA (share / world share):")
print(toy_rca.round(2))
print("\nMcp (1 where RCA > 1):")
print(toy_mcp)
print("\ndiversity (products per country):", toy_diversity.to_dict())
print("ubiquity (countries per product):", toy_ubiquity.to_dict())

Country A and Country B each export two products competitively, yet A makes the rarer motorcycles and cell phones while B makes wheat and textiles, the most common product here. Equal diversity does not mean equal sophistication, and measuring that gap is what the complexity index below is for. Everything below repeats these five steps on the real data.

## 1. Inspect the basket

Before computing anything, look at what Morocco actually sells to the world. This first pass surfaces what you already believe, so you can test it against the numbers.

### Predict-before-run #1

Commit an answer before running the next cell. Say it to your neighbour or write it down.

> What is Morocco's single largest export product, by value, in 2023?

Most people say phosphates or fertilizer. Hold that guess and check it against the data.

In [ ]:
# Morocco's ten largest exports in 2023, with human-readable product names.
one_country_year = exports.query("Unit == @COUNTRY and Period == @YEAR")
top10 = (
    one_country_year.merge(fields, on="Field ID")
    .nlargest(10, "Outputs")
    .assign(share=lambda d: d["Outputs"] / one_country_year["Outputs"].sum())
)
top10[["Field Name", "Field ID", "Outputs", "share"]].reset_index(drop=True)

### Morocco's largest exports

Morocco's top exports are Passenger Cars, Electric Wire and Cable, and Mixed Fertilizers. The fertilizer is the processed product, and the raw rock (Natural Phosphates) ranks only ninth. Many people picture a phosphate economy, while the data points to automotive and processed chemicals. That gap between reputation and measured exports is why we measure complexity instead of trusting reputation.

### Morocco's export basket as a treemap

Each rectangle is a product, sized by export value and coloured by domain, in the style of the Atlas. Hover to read values. This is the one interactive figure today. Every other figure is matplotlib, so the notebook still reads offline if the WiFi fails.

In [ ]:
from IPython.display import HTML  # noqa: E402  (kept next to its one use, for clarity)

treemap_df = (
    one_country_year.merge(fields, on="Field ID")
    .query("Outputs > 0")
    .sort_values("Outputs", ascending=False)
    .head(60)  # top 60 products keeps the picture legible
)

fig = px.treemap(
    treemap_df,
    path=[px.Constant(f"{COUNTRY} exports {YEAR}"), "Domain Name", "Field Name"],
    values="Outputs",
    color="Domain Name",
    title=f"{COUNTRY} export basket, {YEAR} (sized by value, coloured by domain)",
)
fig.update_traces(root_color="lightgrey")
fig.update_layout(margin=dict(t=50, l=10, r=10, b=10))

# We render the treemap as a self-contained HTML block with plotly.js embedded
# inline (include_plotlyjs="inline"), not via fig.show(). Here is why. fig.show()
# stores only a plotly-JSON mimetype, which nbconvert cannot render, so the
# treemap would vanish from the offline reference HTML (the WiFi-fail fallback)
# even though it is the headline figure. Embedding the library inline makes the
# one interactive exhibit survive export and work with no internet. matplotlib
# keeps every other figure light; see the spec's rendering split.
HTML(fig.to_html(full_html=False, include_plotlyjs="inline"))

## 2. Normalize: from dollars to shares

We make countries comparable by removing size. The USA and Morocco both export cars, but the USA exports far more of everything. To compare specialization we use each country's export shares, the fraction of its basket each product makes up.

First we reshape the long table into a country by product matrix, the fundamental object of complexity analysis. Rows are countries, columns are products, each cell an export value. Everything from here is linear algebra on this matrix.

In [ ]:
# Pivot to a country (row) x product (column) matrix of export values for YEAR.
# fill_value=0 because a country that doesn't export a product has value 0, not
# missing. That is a modeling choice we rely on below.
year_long = exports.query("Period == @YEAR")[["Unit", "Field ID", "Outputs"]]
X = year_long.pivot_table(index="Unit", columns="Field ID", values="Outputs", fill_value=0.0)
print(f"matrix shape: {X.shape[0]} countries x {X.shape[1]} products")
X.iloc[:3, :4]

Now we compute the shares, the simplest step of the day. Each country's row divides by that country's own total exports, so every row sums to 1. Read it carefully, because the fix-the-bug cell below plays against it.

In [ ]:
# Export shares: each cell = product's value / that country's TOTAL exports.
# axis=1 division means "divide each row by its own row sum".
country_totals = X.sum(axis=1)
shares = X.div(country_totals, axis=0)

# Sanity peek: Morocco's share rows should sum to 1.
print("Morocco's shares sum to:", round(shares.loc[COUNTRY].sum(), 6))
shares.loc[COUNTRY].nlargest(5)

In [ ]:
# Self-check: shares must sum to 1 for EVERY country (an invariant of the concept,
# not a memorized number). If this fails you probably divided by the wrong axis.
assert (shares.sum(axis=1) - 1).abs().max() < 1e-9, (
    "shares must sum to 1 per country. Divide by each country's row total "
    "(axis=0), not by the column totals."
)
print("OK: every country's shares sum to 1.")

### Fix the broken cell: the classic normalization bug

The cell below tries to compare two countries' export profiles but compares raw values instead of shares, so it only rediscovers that the big economy is bigger. Repair the line marked `# TODO(core)` so the comparison is on shares, then run the self-check.

This bug lives on a single pair of country vectors, so it is not the `world_share` expression you complete next. The lesson is the concept, comparing shares rather than raw values, which you then apply yourself.

In the live session the instructor types the wrong version first, runs it, and reads the nonsense aloud: why does the USA beat Morocco on everything? Watching the bug get caught teaches why normalization exists.

In [ ]:
# Goal: which products is Morocco MORE focused on than the USA, relative to each
# country's own basket? That is a comparison of SHARES, not of raw dollars.
ma_values = X.loc[COUNTRY]  # Morocco's raw export values, per product
us_values = X.loc["US"]  # the USA's raw export values, per product

# TODO(core): the two lines below compare RAW VALUES, and that is the bug. Replace
# them so each country's vector is turned into SHARES (each product as a fraction
# of THAT country's total exports) BEFORE you take the ratio. The variable
# `focus_ratio` should then be Morocco's share ÷ the USA's share, per product.
#
# PROMPT: divide each vector by its OWN sum first (ma_values / ma_values.sum()),
# then take the ratio of the two share vectors.
# TODO(core): write your code here
raise NotImplementedError()

print("Products Morocco is MOST focused on relative to the USA (by share ratio):")
print(focus_ratio.replace([np.inf, -np.inf], np.nan).dropna().nlargest(5))
# Diagnosis (say it aloud): comparing raw values just says "the USA is bigger."
# Only after dividing by each country's own total does "focus" mean anything.

In [ ]:
# Self-check that inspects the variable YOU edited (`focus_ratio`), not a fresh
# recompute off to the side. The bug was: focus_ratio = ma_values / us_values
# (raw values). The fix is the ratio of the two SHARE vectors. We test that:
#   (1) focus_ratio equals the correct share-ratio, cell for cell; and
#   (2) focus_ratio is NOT still the raw-value ratio (so leaving the bug in place
#       fails, even if you deleted the error and defined focus_ratio as anything).
# This is what makes the fix-the-bug exercise actually test the fix.
_share_ratio = (ma_values / ma_values.sum()) / (us_values / us_values.sum())
_raw_ratio = ma_values / us_values
# Compare only where both are finite (a 0/0 or x/0 gives nan/inf in both, equally).
_mask = np.isfinite(_share_ratio.values) & np.isfinite(focus_ratio.values)
assert np.allclose(focus_ratio.values[_mask], _share_ratio.values[_mask]), (
    "focus_ratio must be Morocco's SHARE divided by the USA's SHARE, per product. "
    "Divide each country's vector by its OWN total FIRST, then take the ratio. "
    "That is the fix for the raw-values bug."
)
# And it must differ from the raw-value ratio (the bug), so the unfixed cell fails.
_raw_mask = np.isfinite(_raw_ratio.values) & np.isfinite(focus_ratio.values)
assert not np.allclose(focus_ratio.values[_raw_mask], _raw_ratio.values[_raw_mask]), (
    "focus_ratio is still the RAW-value ratio (ma_values / us_values), which is the "
    "bug. Convert each vector to shares before dividing."
)
print("OK: focus_ratio is the scale-free share-ratio, not the raw-values bug.")

## 3. Revealed comparative advantage (RCA)

We compare each share to the world, so that "specialized" has meaning. A 5% share is impressive for a rare product and unremarkable for a common one. RCA, due to Balassa, divides a country's share of a product by the world's share of that product:

$$ \text{RCA}_{cp} = \frac{\text{country } c\text{'s share of product } p}
                           {\text{world's share of product } p} $$

RCA above 1 means the country is more specialized in the product than the world average. This cell is mostly written for you; only the world-share denominator is blank.

In [ ]:
# COMPLETION exercise: fill in ONLY the world-share denominator on the blanked
# line. Everything else in this cell is already written for you, a one-line
# completion rather than a blank cell, because you have the `shares` schema fresh
# from the step above.
#
# PROMPT: `world_share` should be each product's share of TOTAL world exports,
# i.e. the column sum of X (sum over countries, axis=0) divided by the grand
# total of X. The next line (the RCA division itself) is already done for you.
# TODO(core): complete the one missing line below.
rca = shares.div(world_share, axis=1)  # worked for you: share ÷ world share

print(f"{COUNTRY}'s RCA for Passenger Cars (P - 8703):", round(rca.loc[COUNTRY, "P - 8703"], 2))
rca.loc[COUNTRY].nlargest(5)

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1, the shape. `world_share` has one number per product, one per column of `X`, so you sum down each column. Which axis sums down columns, `axis=0` or `axis=1`?

Hint 2, the share. A share is a part over a whole. The part is each product's world total, `X.sum(axis=0)`. The whole is all world exports, `X.sum().sum()`.

Still stuck? Then `world_share = X.sum(axis=0) / X.sum().sum()`. Make sure you see why that is a per-product world share before moving on.
</details>

In [ ]:
# Self-check: directional + STRUCTURAL invariants. The directional one alone
# (cars > 1) is too weak, since a constant matrix like rca = 2 everywhere would
# pass it. So we also assert two things a constant cannot fake:
#   (1) Balassa identity: weighting each country's RCA for a product by that
#       country's share of WORLD exports and summing over countries gives ~1 for
#       every product (RCA is a share-of-shares; this is true by construction).
#   (2) Real spread: RCA must range from well below 1 to well above 5, since a
#       country both under- and over-specializes. A constant fill has zero spread.
assert rca.loc[COUNTRY, "P - 8703"] > 1, (
    "Morocco over-specializes in Passenger Cars, so RCA there should be > 1. "
    "Did you divide each country's SHARE by the product's world share?"
)
_world_weights = X.sum(axis=1) / X.sum().sum()  # each country's share of world exports
_balassa = rca.mul(_world_weights, axis=0).sum(axis=0)  # weighted mean of RCA per product
assert (_balassa.dropna() - 1).abs().max() < 1e-6, (
    "the world-export-weighted mean of RCA across countries must equal 1 for every "
    "product (the Balassa identity). A constant or hand-typed RCA cannot satisfy "
    "this. Re-check that you divided each country's SHARE by the product's world share."
)
assert rca.values.min() < 1 and rca.values.max() > 5, (
    "RCA must show real spread, some entries far below 1 (under-specialized) and "
    "some far above 5 (strongly specialized). A constant value has no spread."
)
print("OK: Morocco's RCA in Passenger Cars is", round(rca.loc[COUNTRY, "P - 8703"], 2))
print("OK: Balassa identity holds and RCA shows real spread.")

### Interpret (write two or three sentences, no AI, this is for you)

Look at your own `rca.loc[COUNTRY].nlargest(5)` above. Pick the one product a Moroccan minister is most likely to point at and say "the data proves we should build more of these plants." Now make the counter-argument from the same number. What does this RCA value actually tell the minister, and what does it leave unsaid about whether to invest there next? Ground your answer in the specific product and figure on your screen.

TODO(concept): your answer here.

## 4. Binarize: the Mcp matrix (present or not present)

We turn a continuous value into a yes-or-no answer: does this country make this product competitively? We collapse RCA into a 0/1 matrix Mcp, equal to 1 where RCA > 1 and 0 otherwise. Throwing away magnitude to measure breadth is a real modeling choice, and it is the conceptual heart of today, so this step is fully on you.

### Predict-before-run #2

Commit an answer. Of the roughly 862 products in the data, how many does Morocco export competitively (RCA > 1): about 5, 50, or 500? Write your guess, then build Mcp and count.

In [ ]:
# TODO(core): build the binary presence matrix `mcp` from `rca`.
#
# PROMPT: mcp should be 1.0 where rca > 1 and 0.0 otherwise. Comparing a DataFrame
# to 1 gives booleans; convert to float so we can do linear algebra later.
# TODO(core): write your code here
raise NotImplementedError()

print("Morocco exports competitively (RCA>1) in", int(mcp.loc[COUNTRY].sum()), "products.")
mcp.iloc[:3, :6]

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1, the rule. Competitive presence means RCA above the Balassa threshold of 1. So you need True/False for every cell of `rca`: is it greater than 1?

Hint 2, boolean to number. `rca > 1` gives a True/False DataFrame. We want floats so we can do matrix algebra later, and `.astype(float)` turns True/False into 1.0 and 0.0.

Still stuck? Then `mcp = (rca > 1).astype(float)`. Confirm you see why the threshold is 1, which is the same share as the world.
</details>

In [ ]:
# Self-check that pins Mcp to the ONE legitimate definition: binarizing YOUR OWN
# `rca` at the Balassa threshold of 1. The decisive check recomputes that
# reference here and demands an EXACT cell-for-cell match.
#
# Why exact-match against a recompute (not bands, not WIPO)? Two shortcuts have to
# fail:
#   - An all-ones / all-zeros / hand-typed constant: fails the exact match (and
#     the binary/cars/aircraft checks below).
#   - Loading WIPO's precomputed "Innovation Capability (Binary)" column instead
#     of binarizing your own RCA: WIPO uses a different presence rule (inverse-HHI,
#     not plain RCA>1), so it disagrees on a few percent of cells and the exact
#     match fails. Copying the answer key does NOT pass.
# Because `rca` itself was already pinned by the Balassa-identity check above, this
# anchors the whole hand-roll spine to the method, not to a shipped answer.
mcp_ref = (rca > 1).astype(float)  # the only correct Mcp: binarize your own RCA
assert mcp.shape == mcp_ref.shape, (
    "Mcp must have the same country x product shape as `rca`, since it is `rca` "
    "binarized, not a matrix from another source."
)
mcp_aligned = mcp.reindex(index=mcp_ref.index, columns=mcp_ref.columns)
assert np.array_equal(mcp_aligned.values, mcp_ref.values), (
    "Mcp must equal `(rca > 1).astype(float)` exactly, cell for cell. If this "
    "fails you did NOT binarize your own RCA at the threshold 1. For example you "
    "loaded WIPO's precomputed capability column (a different presence rule) or "
    "filled a constant. Compute Mcp yourself from `rca`."
)
# The same invariants as before, kept because their error messages teach the why:
#   (1) binary values only;
#   (2) a known PRESENT product (cars) is 1 AND a known ABSENT one (aircraft,
#       P - 8802) is 0;
#   (3) Morocco's competitive count is in a plausible band (not 0, not all 862);
#   (4) the global density of 1s is well under 50%.
assert set(np.unique(mcp.values)) <= {0.0, 1.0}, (
    "Mcp must contain only 0s and 1s. Apply the rule RCA > 1 and cast to float."
)
assert mcp.loc[COUNTRY, "P - 8703"] == 1.0, (
    "Morocco's RCA in cars is > 1, so Mcp there must be 1. Check your threshold."
)
assert mcp.loc[COUNTRY, "P - 8802"] == 0.0, (
    "Morocco does NOT export aircraft (P - 8802) competitively, so Mcp there must "
    "be 0. If it is 1, you may have filled a constant instead of applying RCA > 1."
)
_ma_count = int(mcp.loc[COUNTRY].sum())
assert 80 <= _ma_count <= 200, (
    f"Morocco should export competitively in a plausible 80-200 products; got "
    f"{_ma_count}. A value of 0 or {mcp.shape[1]} means the threshold wasn't applied."
)
assert mcp.values.mean() < 0.5, (
    f"global density of competitive presence is {mcp.values.mean():.1%}; it must be "
    "well under 50% (most country-product cells are 0). An all-ones matrix fails this."
)
print("OK: Mcp exactly matches RCA>1, is binary, and count/density are sane.")

### Interpret (no AI)

From your own `rca.loc[COUNTRY]`, find one product where Morocco's RCA is very high, say above 10, and one that just clears the bar, a little above 1. After binarizing, both became 1. Name the two products with their RCA values from your screen, say what we threw away by collapsing them, and argue why that loss is acceptable when the next steps measure the breadth of what Morocco can do rather than how dominant it is in any one product.

TODO(concept): your answer here.

## 5. Count: diversity and ubiquity

We read the matrix two ways. Summing across a row gives diversity, the number of products a country makes competitively. Summing down a column gives ubiquity, the number of countries that make a product competitively. These two counts are the raw material of the complexity index.

In [ ]:
# TODO(core): diversity = how many products each COUNTRY makes competitively.
#
# PROMPT: sum mcp ACROSS each row (axis=1), which counts the 1s in each country's
# row.
# TODO(core): write your code here
raise NotImplementedError()

print(f"{COUNTRY} diversity (products made competitively):", int(diversity.loc[COUNTRY]))
diversity.nlargest(5)

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1, the direction. Diversity is per country, one number per row of `mcp`. To collapse a row to one number you sum across the columns.

Hint 2, the axis. In pandas, `axis=1` operates across columns along each row, while `axis=0` operates down rows. You want the one that leaves one value per country.

Still stuck? Then `diversity = mcp.sum(axis=1)`. Sanity check: a diversified economy should beat a one-product oil exporter.
</details>

In [ ]:
# Self-check: diversity is a non-negative count, and a diversified economy
# (Morocco) must beat a highly concentrated one (Chad, TD, an oil/few-goods
# exporter). This is an invariant of what "diversity" means, not a pinned number.
assert (diversity >= 0).all(), "diversity is a count of products; it cannot be negative."
assert diversity.loc[COUNTRY] > diversity.loc["TD"], (
    "a diversified economy (Morocco) should be more diverse than a concentrated "
    "one (Chad). Did you sum across rows (axis=1)?"
)
print("OK: diversity looks right (MA > TD).")

In [ ]:
# TODO(core): ubiquity = how many COUNTRIES make each product competitively.
#
# PROMPT: sum mcp DOWN each column (axis=0), which counts the 1s in each product's
# column. This parallels diversity on the other axis: row versus column is the
# whole mental model.
# TODO(core): write your code here
raise NotImplementedError()

print("Most ubiquitous products (made competitively by the most countries):")
ubiquity.nlargest(5)

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1, the mirror of diversity. Ubiquity is per product, one number per column of `mcp`. To collapse a column to one number you sum down the rows.

Hint 2, the axis. If diversity used `axis=1` across columns, ubiquity uses the other axis to count down each column. That flip between row and column is the whole mental model of Mcp.

Still stuck? Then `ubiquity = mcp.sum(axis=0)`. Sanity check: there is exactly one ubiquity value per product, one per column of `mcp`.
</details>

In [ ]:
# Self-check: ubiquity is a non-negative count, one entry per product.
assert (ubiquity >= 0).all(), "ubiquity is a count of countries; it cannot be negative."
assert len(ubiquity) == mcp.shape[1], "ubiquity should have one value per product (column)."
print("OK: ubiquity computed for every product.")

### Diversity versus average ubiquity

A light plot to relieve the load before the hardest step. Each point is a country: its diversity against the average ubiquity of the products it makes competitively. The downward tilt is the seed of the complexity idea, since the most diversified countries tend to make the rarer, less ubiquitous products.

In [ ]:
# For each country, the mean ubiquity of the products it makes competitively
# (Mcp == 1). This is the first step of the method of reflections, and the
# empirical relationship it formalizes.
avg_ubiquity = pd.Series(
    {c: ubiquity[mcp.loc[c] == 1].mean() if (mcp.loc[c] == 1).any() else np.nan for c in mcp.index}
)
div_ubi = pd.DataFrame({"diversity": diversity, "avg_ubiquity": avg_ubiquity}).dropna()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(div_ubi["diversity"], div_ubi["avg_ubiquity"], s=10, alpha=0.4)

# Simple OLS regression line, the openalex descriptive idiom.
m, b = np.polyfit(div_ubi["diversity"], div_ubi["avg_ubiquity"], 1)
xs = np.linspace(div_ubi["diversity"].min(), div_ubi["diversity"].max(), 100)
ax.plot(xs, m * xs + b, color="black", lw=1.5, label="fit")

# Flag Morocco and its comparators, as on the ECI scatter.
for code in [COUNTRY, *COMPARATORS]:
    if code in div_ubi.index:
        ax.scatter(
            div_ubi.loc[code, "diversity"],
            div_ubi.loc[code, "avg_ubiquity"],
            s=90,
            color="crimson" if code == COUNTRY else "darkorange",
            zorder=5,
        )
        ax.annotate(
            code,
            (div_ubi.loc[code, "diversity"], div_ubi.loc[code, "avg_ubiquity"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=10,
        )
ax.set_xlabel("Diversity (products a country makes competitively)")
ax.set_ylabel("Average ubiquity of those products")
ax.set_title("Diversified countries make less ubiquitous products")
ax.legend()
plt.tight_layout()
plt.show()

### Sort the matrix and the triangle appears

Take a legible slice of the real Mcp matrix and shuffle it, and it looks like noise. Sort the rows by diversity and the columns by ubiquity, and a triangle emerges: diverse countries in the top rows make almost everything, while the products only they make sit in the sparse corner. That nested shape is in the data, not imposed by the method.

In [ ]:
# A legible slice of the real Mcp: spread countries across the diversity range and
# products across the ubiquity range so the pattern is visible in a small block.
c_rank = diversity[diversity > 0].sort_values(ascending=False).index
p_rank = ubiquity[ubiquity > 0].sort_values(ascending=False).index
c_pick = c_rank[np.linspace(0, len(c_rank) - 1, 16).astype(int)]
p_pick = p_rank[np.linspace(0, len(p_rank) - 1, 22).astype(int)]
block = mcp.loc[c_pick, p_pick]
block = block.loc[block.sum(axis=1) > 0, block.sum(axis=0) > 0]  # drop empty rows/cols

# Left panel: rows and columns in random order (looks like noise).
rng = np.random.default_rng(0)
shuffled = block.iloc[rng.permutation(block.shape[0]), rng.permutation(block.shape[1])]

# Right panel: rows sorted by diversity, columns by ubiquity (the triangle).
sorted_block = block.loc[
    block.sum(axis=1).sort_values(ascending=False).index,
    block.sum(axis=0).sort_values(ascending=False).index,
]

# Short product labels: the full names are too long to read sideways.
prod_name = fields.set_index("Field ID")["Field Name"]


def _short(label: str) -> str:
    return label if len(label) <= 18 else label[:17] + "."


fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, panel, title in [
    (axes[0], shuffled, "Shuffled: looks like noise"),
    (axes[1], sorted_block, "Sorted by diversity and ubiquity: a triangle"),
]:
    ax.imshow(panel.values, cmap="Greys", aspect="auto", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_yticks(range(panel.shape[0]))
    ax.set_yticklabels(panel.index, fontsize=7)
    ax.set_xticks(range(panel.shape[1]))
    ax.set_xticklabels([_short(prod_name[c]) for c in panel.columns], rotation=90, fontsize=6)
plt.tight_layout()
plt.show()

## Productive-failure beat (about 2 minutes, then we consolidate)

Before we build the index, and before you read on: how would you measure how sophisticated a country's export basket is? Sketch a formula or rule with your neighbour. Commit to something concrete first, because the struggle is the point. Common first tries are a count of products, total export dollars, or the share of high-tech goods, and none on its own captures sophistication. The next section gives the idea that does.

### Interpret (no AI)

Now that you have your own attempt, sharpen one of the naive rules. Look at your own `ubiquity.nlargest(5)`, the most common products, and at the ubiquity of two Morocco products: Passenger Cars (`P - 8703`) and Phosphoric Acid (`P - 2809`). Print `ubiquity[["P - 8703", "P - 2809"]]` if you like. One is made by many countries, one by few, yet both are hard to make. Using these numbers, explain why ranking products by ubiquity alone mis-measures how sophisticated Morocco's basket is. Do not state the fix; pin down precisely where ubiquity alone goes wrong.

TODO(concept): your answer here.

## 6. Iterate: ECI and PCI via the method of reflections

We combine diversity and ubiquity recursively into one number per country (ECI) and one per product (PCI). This is the linear algebra from the morning L3 lecture, applied. It is the hardest step of the day, so it is fully worked. Read it, run it, and use the notation table and the Parsons exercise to anchor why each line is there.

### Notation, meaning, and the variable used here

| Symbol | Meaning | Variable here |
|---|---|---|
| $M_{cp}$ | 1 if country c makes product p competitively | `mcp` |
| $k_c^0$ | diversity (row sums of $M_{cp}$) | `diversity` |
| $k_p^0$ | ubiquity (column sums of $M_{cp}$) | `ubiquity` |
| $\widetilde{M}_{cc'}$ | country-country reflection matrix | `Mcc` |
| $\widetilde{M}_{pp'}$ | product-product reflection matrix | `Mpp` |
| ECI | standardized 2nd eigenvector over countries | `eci` |
| PCI | standardized 2nd eigenvector over products | `pci` |

### The second eigenvector

The reflection matrices are row-stochastic, so their first eigenvector is the trivial all-ones constant that carries no information. The second eigenvector is the one that ranks countries and products by sophistication. Shipping the first by mistake is the classic off-by-one bug, which the self-check below guards against.

### Parsons exercise (concept, no free-typing)

The next cell assembles the index in eight steps, listed out of order below. Before running, put them in the correct logical order. The code is already written correctly; your job is to understand the sequence, not to type numpy.

1. project back to countries: `kc = mcp1 @ kp`
2. row-normalize: `mcp1 = mcp / diversity`
3. eigendecompose `Mpp`
4. form `Mcc = mcp1 @ mcp2.T` and `Mpp = mcp2.T @ mcp1`
5. column-normalize: `mcp2 = mcp / ubiquity`
6. sign-fix to diversity, then z-score
7. take the 2nd eigenvector as PCI (`kp`)
8. (the 1st eigenvector is the trivial constant, so skip it)

<details>
<summary><b>Reveal the correct order (try it yourself first).</b></summary>

Correct order: 2, 5, 4, 3, 8, 7, 1, 6. Normalize both ways, form the reflection matrices, eigendecompose, skip the trivial first eigenvector, take the second as PCI, project back to countries, then sign-fix and standardize.
</details>

In [ ]:
# === WORKED matrix assembly (read along; the instructor types this live). ======
# This is the highest-intrinsic-load block, so the matrix algebra is fully worked
# and you do NOT free-type numpy here. The two conceptual decisions the Parsons
# exercise and the morning lecture turn on, (1) which eigenvector is the index and
# (2) the sign anchoring, are pulled into the next cell as a small TODO you author
# yourself, so the hard step is not purely given to you.
#
# Drop products/countries that are entirely absent, so we never divide by zero.
present_c = diversity[diversity > 0].index
present_p = ubiquity[ubiquity > 0].index
mcp_r = mcp.loc[present_c, present_p]
div_r = mcp_r.sum(axis=1)
ubi_r = mcp_r.sum(axis=0)

# Method of reflections, assembled in logical order (the Parsons sequence above):
mcp1 = mcp_r.div(div_r, axis=0)  # row-normalize by diversity
mcp2 = mcp_r.div(ubi_r, axis=1)  # column-normalize by ubiquity
Mcc = mcp1.values @ mcp2.values.T  # country-country reflections
Mpp = mcp2.values.T @ mcp1.values  # product-product reflections

# Eigendecompose, then sort the eigenvalues largest-first. `order[0]` indexes the
# largest eigenvalue's eigenvector, `order[1]` the second-largest, and so on.
eigvals, eigvecs = np.linalg.eig(Mpp)
order = np.argsort(-eigvals.real)  # sort eigenvalues largest-first

In [ ]:
# TODO(core): make the TWO decisions that turn the eigenvectors into the index.
#
# (1) PICK THE RIGHT EIGENVECTOR. `order[0]` is the largest eigenvalue (the
#     trivial all-ones constant, no information). `order[1]` is the SECOND. Set
#     `kp` to the real part of the eigenvector column for the eigenvector that
#     actually ranks products by sophistication. (Which index, 0 or 1?)
# (2) SIGN-FIX. The eigensolver returns an arbitrary sign. Set `sign` to +1 if
#     ECI already rises with diversity, else -1, so high-ECI means more-diverse.
#     Use the sign of the correlation between `div_r.values` and `kc`.
#
# PROMPT: kp = eigvecs[:, order[?]].real ; then kc = mcp1.values @ kp is given;
# then sign = 1 if corr(div_r, kc) > 0 else -1.
# TODO(core): write your code here
raise NotImplementedError()

eci_raw = sign * kc
pci_raw = sign * kp

# Standardize EACH index by ITS OWN mean and sd (a port bug in the inherited R
# script standardized PCI using ECI's moments, fixed here).
eci = pd.Series((eci_raw - eci_raw.mean()) / eci_raw.std(), index=present_c)
pci = pd.Series((pci_raw - pci_raw.mean()) / pci_raw.std(), index=present_p)

print(f"{COUNTRY} ECI (trade-only, hand-rolled):", round(eci.loc[COUNTRY], 3))
print("Highest-PCI products:")
pci.nlargest(5)

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1, the eigenvector. The first eigenvector, `order[0]`, is the trivial all-ones constant, so it carries no ranking. You want the next one. Which index comes after 0?

Hint 2, the sign. `np.corrcoef(div_r.values, kc)[0, 1]` is the correlation between diversity and the raw country score. If it is positive, use +1. If it is negative, flip to -1.

Still stuck? Then:
`kp = eigvecs[:, order[1]].real` ; `kc = mcp1.values @ kp` ;
`sign = 1 if np.corrcoef(div_r.values, kc)[0, 1] > 0 else -1`.
</details>

In [ ]:
# Self-check the off-by-one guard. Three parts, the decisive one tying `kp` to the
# ACTUAL decomposition so a pasted answer cannot sneak through:
#   (1) the FIRST eigenvector must be ~constant (the trivial all-ones);
#   (2) the eigenvector YOU selected as PCI (`kp`) must NOT be that constant,
#       i.e. you actually picked the 2nd, not the 1st;
#   (3) `kp` must genuinely BE the second eigenvector of Mpp, so we check the
#       eigenvector equation Mpp @ kp == lambda_2 * kp holds (residual ~ 0).
#       Parts (1)-(2) alone are too weak: any non-constant vector (e.g. WIPO's
#       PCI reindexed onto these products) passes them. The residual check fails
#       for anything that is not literally the order[1] eigenvector, so pasting
#       WIPO's PCI in place of the eigendecomposition is caught here.
first_vec = eigvecs[:, order[0]].real
assert np.allclose(first_vec, first_vec[0], atol=1e-2), (
    "the 1st eigenvector should be ~constant (the trivial all-ones). If it isn't, "
    "ECI/PCI may be reading the wrong eigenvector."
)
assert not np.allclose(kp, kp[0], atol=1e-2), (
    "your PCI vector `kp` is ~constant, which is the trivial 1st eigenvector. "
    "Pick order[1] (the SECOND), not order[0]."
)
# kp must satisfy Mpp @ kp = lambda_2 * kp (it is an eigenvector of Mpp, not some
# external vector). We normalize the residual by the size of kp so the tolerance
# is scale-free regardless of how kp was scaled.
_lambda2 = eigvals[order[1]].real
_residual = Mpp @ kp - _lambda2 * kp
_resid_norm = np.linalg.norm(_residual) / (np.linalg.norm(kp) + 1e-12)
assert _resid_norm < 1e-6, (
    "your `kp` is not actually the second eigenvector of Mpp. Mpp @ kp must equal "
    f"lambda_2 * kp (got relative residual {_resid_norm:.2e}). Set "
    "`kp = eigvecs[:, order[1]].real` from the decomposition; do NOT paste in a "
    "PCI from another source (e.g. WIPO's), which will not satisfy this equation."
)
# And ECI must be standardized: mean ~0, sd ~1.
assert abs(eci.mean()) < 1e-6 and abs(eci.std(ddof=1) - 1) < 1e-2, (
    "ECI should be standardized (mean 0, sd 1). Did you z-score it?"
)
print("OK: kp is the true 2nd eigenvector of Mpp; ECI is standardized.")

### Interpret (no AI)

(a) You just authored the `sign` line. Run the cell once with `sign` forced to `-1`, then put it back. Describe what happened to Morocco's ECI and to the `pci.nlargest(5)` list, and explain why the sign is arbitrary out of the solver. Then ask whether anchoring the sign to diversity makes ECI the same thing as diversity, or something more.

(b) Pick the top product from your own `pci.nlargest(5)`. Is its high PCI a fact about the chemistry and engineering of that product, or about which countries currently export it? Name one concrete change to the trade data that would move that product's PCI, and say which direction.

TODO(concept): your answer here.

## 7. Validate against WIPO

We check our numbers against an external source and learn that methodology is a choice, not a bug. WIPO's published dataset has its own RCA-based capability matrix and its own PCI. Our hand-rolled numbers should correlate strongly with WIPO's, and they will not match perfectly, because the methods differ.

The concept cell below asks you to name the two methodology differences yourself, so try to reason them out before revealing the answer. We validate with a rank correlation, Spearman's, and expect it below 1.0. An assert demanding exact equality would contradict the lesson.

In [ ]:
# Compare our Mcp to WIPO's binary capability matrix (exports, same year).
wipo_cap = pd.read_parquet(PACKET / "exports" / "capabilities.parquet")
wipo_cap = wipo_cap.query("Period == @YEAR")
wipo_mcp = wipo_cap.pivot_table(
    index="Unit",
    columns="Field ID",
    values="Innovation Capability (Binary)",
    fill_value=False,
).astype(float)

# Align on the countries and products both matrices share, then compare.
rows = mcp.index.intersection(wipo_mcp.index)
cols = mcp.columns.intersection(wipo_mcp.columns)
mine = mcp.loc[rows, cols].values.flatten()
theirs = wipo_mcp.loc[rows, cols].values.flatten()
agreement = (mine == theirs).mean()
print(f"Our Mcp agrees with WIPO's binary capability on {agreement:.1%} of cells.")

In [ ]:
# Compare our PCI to WIPO's "Capability Complexity Index" (the PCI analogue).
wipo_pci = (
    pd.read_parquet(PACKET / "exports" / "field_complexities.parquet")
    .query("Period == @YEAR")
    .set_index("Field ID")["Capability Complexity Index"]
)
common = pci.index.intersection(wipo_pci.index)
pci_rho = spearmanr(pci.loc[common], wipo_pci.loc[common]).statistic
print(f"Spearman(our PCI, WIPO PCI) = {pci_rho:.3f} over {len(common)} products.")

In [ ]:
# Self-check: strong but IMPERFECT rank agreement. The threshold is loose by
# design, since methods differ (see the markdown above). Asserting exact equality
# here would be wrong on purpose.
assert pci_rho > 0.6, (
    f"our PCI should correlate strongly with WIPO's (Spearman > 0.6); got {pci_rho:.3f}. "
    "If it's much lower, re-check the eigenvector and the sign-fix."
)
assert pci_rho < 0.999, (
    "the correlation should NOT be perfect. Our plain RCA>1 and trade-only "
    "eigenvector differs from WIPO's inverse-HHI rule and combined method of "
    "reflections."
)
print(f"OK: strong but imperfect agreement (Spearman = {pci_rho:.3f}), as expected.")

### Interpret (no AI, this is the AI red-light cell)

Your `pci_rho` printed just above, around 0.9: strong but not identical. Before reaching for a debugger, name the two methodology differences that explain the gap, reasoning from what you built: (i) the presence rule you used to decide "competitive", and (ii) whether your index used the trade matrix alone or all four dimensions. Then the part a chatbot cannot do for you. For a policy brief advising the Moroccan government on diversification, which PCI, yours or WIPO's, would you put in the brief, and why, given what each one measures and the audience?

<details>
<summary><b>Reveal the two differences (only after you have named them).</b></summary>

1. Presence rule. We used the plain Balassa rule, RCA > 1. WIPO uses an inverse-HHI rule: a product is present if RCA > 1 or the country is one of the n largest producers, which rescues concentrated activities like patents.
2. Index method. We took a single eigenvector on the trade matrix alone. WIPO computes complexity via the method of reflections across all four dimensions, the combined index, rather than trade alone.
</details>

TODO(concept): your answer here.

## 8. Does complexity track prosperity?

We connect the index to something a policymaker cares about. We plot Morocco's ECI against GDP per capita, with the comparators flagged, and read where Morocco sits relative to the fitted line.

### Predict-before-run #3

Before the scatter, sketch the relationship you expect between a country's ECI and its GDP per capita. Up, down, or flat? And where do you expect Morocco to sit relative to the fitted line, above or below?

In [ ]:
# ECI vs GDP per capita (latest year). One point per country; Morocco highlighted.
gdp_pc = units.query("Period == @YEAR").set_index("Unit")["GDP PC"]
scatter = pd.DataFrame({"eci": eci, "gdp_pc": gdp_pc}).dropna()
scatter = scatter[scatter["gdp_pc"] > 0]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(scatter["gdp_pc"], scatter["eci"], s=18, alpha=0.5)

# Least-squares fit line in log-x space (GDP per capita is roughly log-linear).
logx = np.log(scatter["gdp_pc"])
b1, b0 = np.polyfit(logx, scatter["eci"], 1)
xs = np.linspace(logx.min(), logx.max(), 100)
ax.plot(np.exp(xs), b0 + b1 * xs, color="black", lw=1.5, label="fit")

# Highlight Morocco and its comparators.
for code in [COUNTRY, *COMPARATORS]:
    if code in scatter.index:
        ax.scatter(
            scatter.loc[code, "gdp_pc"],
            scatter.loc[code, "eci"],
            s=90,
            color="crimson" if code == COUNTRY else "darkorange",
            zorder=5,
        )
        ax.annotate(
            code,
            (scatter.loc[code, "gdp_pc"], scatter.loc[code, "eci"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=10,
        )
ax.set_xscale("log")
ax.set_xlabel("GDP per capita (PPP, current international USD, log scale)")
ax.set_ylabel("Trade-only ECI (standardized)")
ax.set_title(f"Complexity tracks prosperity (trade-only ECI vs income, {YEAR})")
ax.legend()
plt.tight_layout()
plt.show()

corr = np.corrcoef(logx, scatter["eci"])[0, 1]
print(f"OLS fit: ECI = {b0:.2f} + {b1:.2f} * ln(GDP per capita)")
print(f"Correlation(log GDP per capita, ECI) = {corr:.3f} (R^2 = {corr**2:.3f})")

In [ ]:
# Self-check: the well-known positive relationship must hold (invariant, not a
# pinned value). A negative or zero correlation would signal a pipeline error.
assert corr > 0.4, (
    f"ECI should correlate positively with GDP per capita; got {corr:.3f}. "
    "A weak/negative value usually means a sign-fix or eigenvector error upstream."
)
print(f"OK: ECI and GDP per capita are positively related (r = {corr:.3f}).")

### Interpret (no AI, AI red-light)

Across countries, ECI rises with income, the fitted line. Morocco sits a little below it here, so its trade-only ECI comes in slightly under what its income predicts. Read that position with care. This is a one-afternoon pipeline on trade data alone, with far less cleaning than the Atlas of Economic Complexity, so the exact distance from the line is not a basis for a growth claim. The takeaway is the positive slope, that more complex economies tend to be richer. Name one reason this simplified pipeline might place a country differently from the Atlas.

TODO(concept): your answer here.

## 9. Recall: close the loop (no scrolling up)

Spaced retrieval. Without looking back, answer from memory. We re-test the same chain at the start of Days 2 and 3, which is where it sticks.

1. In one phrase each, what do Normalize → Benchmark → Binarize → Count → Iterate each do?
2. Why do we use the second eigenvector rather than the first?
3. Why did our PCI not match WIPO's exactly, and was that a bug?

TODO(concept): your answers here.

## Stretch (for the fast finishers, scaffolding removed)

Pick any. These remove the guard-rails: no PROMPT, no self-check.

(a) Method-of-reflections convergence. Instead of the eigenvector, iterate the averaging map by hand: start from `diversity` and `ubiquity`, repeatedly set `kc = mcp1 @ kp` and `kp = mcp2.T @ kc`, standardizing each round. How many iterations pass before the country ranking stops changing? Confirm it converges to the eigenvector ECI.

(b) 2-digit versus 4-digit. Roll the HS92 4-digit products up to their 2-digit chapter (the digits after `P - `) and recompute ECI. How much does Morocco's rank move, and why might the aggregation level matter for a policy claim?

In [ ]:
# (Your stretch code here, no scaffold, no self-check.)

### Stretch: the innovation teaser (hold for Day 5)

The packet has the same five tables for patents, publications, and trademarks, and the pipeline you built is domain-agnostic. Swap `exports/outputs.parquet` for `patents/outputs.parquet` and you get a technological complexity profile instead of a trade one. Try it below, then hold the thought for Day 5, where we ask what Morocco's innovation capabilities say about where it can go.

In [ ]:
# Teaser: same pipeline, different domain. Uncomment and explore.
# patents = pd.read_parquet(PACKET / "patents" / "outputs.parquet")
# (Build shares -> RCA -> Mcp -> diversity/ubiquity -> ECI exactly as above,
#  but on patent FAMILIES instead of export dollars. Hold the interpretation
#  for Day 5.)